# CHEATSHEET EXAMEN — Copy / Paste

Cada bloque es **autocontenido**. Solo cambia:
1. `DATA_PATH` — ruta al CSV
2. `TARGET` — nombre de la columna objetivo

## Orden de uso
1. ¿Predigo un número? → **Bloque A (Regresión)**
2. ¿Predigo una clase? → **Bloque B (Clasificación)**
3. ¿Me piden red neuronal? → **Bloque C (MLP PyTorch)**
4. ¿Hay fechas y no quieren leakage temporal? → **Bloque D (Split temporal)**

## Reglas que NO se rompen
- `fit` **solo** en train. `transform` en val/test.
- Clasificación → `stratify=y` en el split.
- Desbalance → `class_weight="balanced"`.
- Imputar y escalar **dentro** del Pipeline.
- Test se toca **una vez** al final.

## BLOQUE A — Regresión completa (RandomForest)

Pega esta celda, cambia `DATA_PATH` y `TARGET`. Imprime RMSE, MAE y R².

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. CONFIGURAR
DATA_PATH = "datasets/king-county/kc_house_data.csv"
TARGET = "price"
COLS_QUITAR = ["id", "date"]  # IDs y fechas raw fuera. [] si no hay.

# 2. CARGAR
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in COLS_QUITAR if c in df.columns])
print(f"Shape: {df.shape}")

# 3. SPLIT
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 4. PIPELINE (imputa + escala + one-hot, todo dentro)
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

model = Pipeline([
    ("prep", preprocessor),
    ("reg",  RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
])

# 5. ENTRENAR Y EVALUAR
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)
print(f"RMSE: {rmse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"R²:   {r2:.4f}")

## BLOQUE B — Clasificación completa (RandomForest, soporta desbalance)

Vale para binaria y multiclase. Imprime `classification_report` + matriz de confusión.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. CONFIGURAR
DATA_PATH = "datasets/thyroid/thyroidDF.csv"
TARGET = "target"
COLS_QUITAR = []  # quita IDs si los hay

# 2. CARGAR
df = pd.read_csv(DATA_PATH)
df = df.drop(columns=[c for c in COLS_QUITAR if c in df.columns])
df = df.dropna(subset=[TARGET])  # NaN en target sí se quitan
print("Distribución de clases:")
print(df[TARGET].value_counts(normalize=True).round(3))

# 3. SPLIT CON STRATIFY (siempre en clasificación)
X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. PIPELINE
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler",  StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

model = Pipeline([
    ("prep", preprocessor),
    ("clf",  RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",   # ← clave si hay desbalance
        random_state=42,
        n_jobs=-1,
    )),
])

# 5. ENTRENAR Y EVALUAR
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred, zero_division=0))
print("Matriz de confusión:")
print(confusion_matrix(y_test, y_pred))
print(f"F1 macro: {f1_score(y_test, y_pred, average='macro', zero_division=0):.4f}")

## BLOQUE C — MLP PyTorch (red neuronal)

Úsalo si te piden **explícitamente** una red neuronal. Asume que ya corriste el preprocesado del Bloque A o B y tienes `X_train_proc`, `X_test_proc` (matrices numéricas) y `y_train`, `y_test`.

Para obtenerlos:
```python
X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc  = preprocessor.transform(X_test)
```

### C.1 — MLP para REGRESIÓN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. ESCALAR TARGET (clave en regresión: si y vale 100000, los gradientes explotan)
y_scaler = StandardScaler()
y_train_s = y_scaler.fit_transform(np.log1p(y_train.values).reshape(-1, 1)).astype(np.float32)
y_test_s  = y_scaler.transform(np.log1p(y_test.values).reshape(-1, 1)).astype(np.float32)

# 2. DATALOADERS
def loader(X, y, bs=256, shuffle=False):
    X_t = torch.tensor(np.asarray(X), dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.float32)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=bs, shuffle=shuffle)

train_loader = loader(X_train_proc, y_train_s, shuffle=True)
test_loader  = loader(X_test_proc,  y_test_s)

# 3. MODELO
n_input = X_train_proc.shape[1]

class MLPReg(nn.Module):
    def __init__(self, n_input):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),     nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1),       # sin activación = regresión
        )
    def forward(self, x): return self.net(x)

model_nn = MLPReg(n_input).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3)

# 4. ENTRENAR
for epoch in range(50):
    model_nn.train()
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model_nn(X_b), y_b)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}  loss={loss.item():.4f}")

# 5. PREDECIR Y DESESCALAR
model_nn.eval()
preds_s = []
with torch.no_grad():
    for X_b, _ in test_loader:
        preds_s.append(model_nn(X_b.to(device)).cpu().numpy())
preds_s = np.concatenate(preds_s)
y_pred_nn = np.expm1(y_scaler.inverse_transform(preds_s).squeeze())

from sklearn.metrics import mean_squared_error, r2_score
rmse = mean_squared_error(y_test, y_pred_nn) ** 0.5
print(f"Red Neuronal  RMSE={rmse:,.2f}  R²={r2_score(y_test, y_pred_nn):.4f}")

### C.2 — MLP para CLASIFICACIÓN

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, f1_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. CODIFICAR ETIQUETAS A ENTEROS (PyTorch necesita ints, no strings)
le = LabelEncoder()
y_train_i = le.fit_transform(y_train)
y_test_i  = le.transform(y_test)
n_clases = len(le.classes_)
print(f"Clases: {dict(enumerate(le.classes_))}")

# 2. DATALOADERS (etiquetas como long)
def loader_cls(X, y, bs=128, shuffle=False):
    X_t = torch.tensor(np.asarray(X), dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    return DataLoader(TensorDataset(X_t, y_t), batch_size=bs, shuffle=shuffle)

train_loader = loader_cls(X_train_proc, y_train_i, shuffle=True)
test_loader  = loader_cls(X_test_proc,  y_test_i)

# 3. MODELO
n_input = X_train_proc.shape[1]

class MLPCls(nn.Module):
    def __init__(self, n_input, n_clases):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_input, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),      nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, n_clases), # n_clases neuronas, sin softmax
        )
    def forward(self, x): return self.net(x)

model_nn = MLPCls(n_input, n_clases).to(device)

# 4. PESOS DE CLASE PARA DESBALANCE (= class_weight='balanced')
counts = np.bincount(y_train_i)
w = torch.tensor(len(y_train_i) / (n_clases * counts), dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=w)
optimizer = torch.optim.Adam(model_nn.parameters(), lr=1e-3)

# 5. ENTRENAR
for epoch in range(50):
    model_nn.train()
    for X_b, y_b in train_loader:
        X_b, y_b = X_b.to(device), y_b.to(device)
        optimizer.zero_grad()
        loss = criterion(model_nn(X_b), y_b)
        loss.backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}  loss={loss.item():.4f}")

# 6. PREDECIR
model_nn.eval()
all_preds = []
with torch.no_grad():
    for X_b, _ in test_loader:
        logits = model_nn(X_b.to(device))
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())

print(classification_report(y_test_i, all_preds, target_names=le.classes_, zero_division=0))
print(f"F1 macro: {f1_score(y_test_i, all_preds, average='macro', zero_division=0):.4f}")

## BLOQUE D — Split TEMPORAL (cuando hay fechas)

**Síntoma**: el dataset tiene columna `date` o `timestamp`. Si haces `train_test_split` aleatorio, hay leakage temporal.

Sustituye el `train_test_split` del Bloque A por esto:

In [ ]:
# Asume df cargado con columna de fecha
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# Extraer features útiles ANTES de quitar la columna
df["sale_year"]  = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df = df.drop(columns=["date"])

# Split por POSICIÓN (no aleatorio): 80% antiguo → train, 20% reciente → test
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df  = df.iloc[split_idx:]

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=[TARGET])
y_test  = test_df[TARGET]

print(f"Train: {len(X_train):,} filas  |  Test: {len(X_test):,} filas")
print("Test contiene fechas POSTERIORES al train ✓")

## BLOQUE E — Tabla rápida de métricas

### Regresión
| Métrica | Cuándo |
|---|---|
| **RMSE** | Por defecto. Misma unidad que el target. |
| **MAE**  | Si hay outliers (más robusto). |
| **R²**   | Comparar modelos. 1=perfecto, 0=tan malo como la media. |

### Clasificación
| Métrica | Cuándo |
|---|---|
| **Accuracy** | Solo si las clases están **balanceadas**. |
| **F1 macro** | Multiclase desbalanceada (Thyroid). |
| **F2** | Medicina: falsos negativos son peores (perder un enfermo es grave). |
| **Precision** | Falsos positivos son caros (spam: marcar bueno como spam). |
| **Recall** | Falsos negativos son caros (cáncer: no detectarlo). |

### Fórmulas
```
Precision = TP / (TP + FP)
Recall    = TP / (TP + FN)
F1        = 2·P·R / (P + R)
F2        = 5·P·R / (4P + R)   ← recall pesa más
```

## BLOQUE F — Si me preguntan "¿por qué?"

| Pregunta del profe | Respuesta corta |
|---|---|
| ¿Por qué Pipeline? | Para que el preprocesado se haga **solo con stats de train**, sin leakage. |
| ¿Por qué `stratify=y`? | Para mantener la proporción de clases en train y test. |
| ¿Por qué `class_weight='balanced'`? | Para que el modelo no ignore la clase minoritaria. |
| ¿Por qué quitar `id`? | Memoriza filas concretas, no generaliza. |
| ¿Por qué split temporal? | Si hay fechas, no podemos predecir el pasado con el futuro. |
| ¿Por qué escalar y en regresión NN? | Sin escalar, MSE ~10^10 → gradientes explotan. |
| ¿Por qué `model.eval()`? | Desactiva Dropout y BatchNorm pone stats globales. |
| ¿Por qué clásico gana en tabular? | Árboles toleran NaN, no necesitan escalado, capturan interacciones sin tunear arquitectura. |